## Parse Structured XML Generated by Grobid

In [ ]:
import json
import logging
import Levenshtein
import os
import requests

from bs4 import BeautifulSoup

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
TEI_XML_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei/'
TEI_XML_CROSSREF_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/tei_crossref_citations/'
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_full/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_grobid/'

LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD = 5

In [ ]:
def extract_grouped_refs(soup):
    """
    Extract references grouped by occurrence in the same paragraph of the paper.
    """
    text = soup.find('text')
    grouped_refs = []
    for child_tag in text.body.children:
        # Skip empty lines and non-text tags (e.g., notes)
        if child_tag.name != 'div':
            continue

        # Extract title if present
        title = child_tag.head.text if child_tag.head else ''

        # Group all references from a single div
        ref_tags = child_tag.find_all('ref', attrs={"type": "bibr"})
        refs = [(ref_tag.get('target', None), ref_tag.text)
                for ref_tag in ref_tags]
        grouped_refs.append({'title': title, 'references': refs})
    return grouped_refs

In [ ]:
def extract_refs(soup):
    """
    Extract list of references for the paper.
    """
    bibliography = soup.find('listBibl')
    refs = {}
    for bibl_struct in bibliography.children:
        # Skip empty lines
        if bibl_struct.name != 'biblStruct':
            continue

        ref_id = bibl_struct['xml:id']
        ref_text = ''
        if bibl_struct.analytic and bibl_struct.analytic.title:
            ref_text = bibl_struct.analytic.title.text
        refs[ref_id] = ref_text
    return refs

In [ ]:
def parse_xml(filename):
    with open(filename, 'r') as xml_file:
        soup = BeautifulSoup(xml_file, 'xml')
    
    grouped_refs = extract_grouped_refs(soup)
    refs = extract_refs(soup)
    return grouped_refs, refs

In [ ]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [ ]:
def process_xml(filename):
    pmid = filename.split('.')[0]
    xml_raw = os.path.join(TEI_XML_FOLDER, filename)
    xml_crossref = os.path.join(TEI_XML_CROSSREF_FOLDER, filename)

    grouped_refs_raw, refs_raw = parse_xml(xml_raw)
    grouped_refs_crossref, refs_crossref = parse_xml(xml_crossref)
    
    if grouped_refs_raw != grouped_refs_crossref:
        print('Grouped refs raw vs crossref differ')
        
    refs_merged = {}
    for ref_id, ref_raw in refs_raw.items():
        ref_fixed = refs_crossref[ref_id]
        distance = Levenshtein.distance(ref_raw.lower(), ref_fixed.lower())
        
        selected = False
        reason = ''
        if distance > LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD and ref_raw:
            selected = True
            reason = ref_raw
        merged_ref = {
            'title': ref_fixed,
            'selected': selected,
            'reason': reason
        }

        refs_merged[ref_id] = merged_ref
    
    grouped_refs_filename = os.path.join(GROUPED_REFS_FOLDER, f'{pmid}.json')
    export_to_json(grouped_refs_raw, grouped_refs_filename)
        
    refs_filename = os.path.join(REFS_FOLDER, f'{pmid}.json')
    # Skip merging
    export_to_json(refs_raw, refs_filename)

In [ ]:
for folder in [GROUPED_REFS_FOLDER, REFS_FOLDER]:
    if not os.path.exists(folder):
        os.mkdir(folder)
for filename in os.listdir(TEI_XML_FOLDER):
    if filename.endswith('.xml'):
        logging.info(f'Processing {filename}')
        process_xml(filename)

## Validating Grouped References

In [73]:
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_validated/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_validated/'

In [69]:
def flatten_grouped_refs(grouped_refs):
    refs = []
    for el in grouped_refs:
        if isinstance(el, dict):
            refs.extend(flatten_grouped_refs(el['references']))
        else:
            assert len(el) == 2
            # Clean numeric ids, e.g. 'REF. 37' -> 37
            grobid_id, numeric_id = el
            numeric_id = int(''.join(list(filter(lambda c: c.isdigit(), numeric_id))))
            # Check Grobid ID: should be '#bXXX'
            if grobid_id:
                assert grobid_id[:2] == '#b'
                grobid_number = int(grobid_id[2:])  # should be a valid int
            refs.append([grobid_id, numeric_id])
    return refs

### 1. JSON is valid - OK

In [75]:
import json
import os

for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    print(file, end='\t')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)
    print('OK')

26580716.json	OK
26580717.json	OK
26656254.json	OK
26667849.json	OK
26675821.json	OK
26678314.json	OK
26688349.json	OK
26688350.json	OK
27677859.json	OK
27677860.json	OK
27834397.json	OK
27834398.json	OK
27890914.json	OK
27904142.json	OK
27916977.json	OK
28003656.json	OK
28792006.json	OK
28852220.json	OK
28853444.json	OK
28920587.json	OK
29147025.json	OK
29170536.json	OK
29213134.json	OK
29321682.json	OK
30108335.json	OK
30390028.json	OK
30459365.json	OK
30467385.json	OK
30578414.json	OK
30644449.json	OK
30679807.json	OK
30842595.json	OK
31686003.json	OK
31806885.json	OK
31836872.json	OK
31937935.json	OK
32005979.json	OK
32020081.json	OK
32042144.json	OK
32699292.json	OK


In [74]:
for file in sorted(os.listdir(REFS_FOLDER)):
    print(file, end='\t')
    filename = os.path.join(REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)
    print('OK')

26580716.json	OK
26580717.json	OK
26656254.json	OK
26667849.json	OK
26675821.json	OK
26678314.json	OK
26688349.json	OK
26688350.json	OK
27677859.json	OK
27677860.json	OK
27834397.json	OK
27834398.json	OK
27890914.json	OK
27904142.json	OK
27916977.json	OK
28003656.json	OK
28792006.json	OK
28852220.json	OK
28853444.json	OK
28920587.json	OK
29147025.json	OK
29170536.json	OK
29213134.json	OK
29321682.json	OK
30108335.json	OK
30390028.json	OK
30459365.json	OK
30467385.json	OK
30578414.json	OK
30644449.json	OK
30679807.json	OK
30842595.json	OK
31686003.json	OK
31806885.json	OK
31836872.json	OK
31937935.json	OK
32005979.json	OK
32020081.json	OK
32042144.json	OK
32699292.json	OK


### 2. Looking for Errors in Grouped References JSONs

* null / None values of Grobid ID (`#bXXX`) occur if reference was not found by Grobid
* Grobid ID and numeric reference ID should correspond one to another

In [71]:
import json
import os

print('\t\tNO_NULL_IDS\tUNIQUE_IDS')
for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    print(file, end='\t')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        grouped_refs = json.load(f)
    flat_grouped_refs = flatten_grouped_refs(grouped_refs)
    
    # Check that no 'null' refs are present in grouped reference files (might occur due to Grobid errors)
    null_refs = False
    for ref in flat_grouped_refs:
        # Check if flatten function works
        assert type(ref) == list
        assert len(ref) == 2
        grobid_id, numeric_id = ref
        if not grobid_id:
            null_refs = True
            break
    if null_refs:
        print('ERROR', end='\t\t')
    else:
        print('OK', end='\t\t')
        
    # Check that grobid_id <-> numeric_id is a bijection
    grobid2numeric = {}
    numeric2grobid = {}
    for ref in flat_grouped_refs:
        grobid_id, numeric_id = ref
        
        if grobid_id not in grobid2numeric:
            grobid2numeric[grobid_id] = set()
        grobid2numeric[grobid_id].add(numeric_id)
        
        if numeric_id not in numeric2grobid:
            numeric2grobid[numeric_id] = set()
        numeric2grobid[numeric_id].add(grobid_id)
    
    unique_ids = True
    error_ids = []
    for gid, nids in grobid2numeric.items():
        if gid == 'null':
            continue
        if len(nids) > 1:
            error_ids.append((gid, nids))
    for nid, gids in numeric2grobid.items():
        if len(gids) > 1:
            error_ids.append((nid, gids))
    if unique_ids:
        print('OK')
    else:
        print(f"ERROR {', '.join([error_ids])}")

		NO_NULL_IDS	UNIQUE_IDS
26580716.json	OK		OK
26580717.json	OK		OK
26656254.json	OK		OK
26667849.json	OK		OK
26675821.json	OK		OK
26678314.json	OK		OK
26688349.json	OK		OK
26688350.json	OK		OK
27677859.json	OK		OK
27677860.json	OK		OK
27834397.json	OK		OK
27834398.json	ERROR		OK
27890914.json	OK		OK
27904142.json	OK		OK
27916977.json	OK		OK
28003656.json	OK		OK
28792006.json	OK		OK
28852220.json	OK		OK
28853444.json	OK		OK
28920587.json	OK		OK
29147025.json	OK		OK
29170536.json	OK		OK
29213134.json	OK		OK
29321682.json	OK		OK
30108335.json	OK		OK
30390028.json	OK		OK
30459365.json	OK		OK
30467385.json	OK		OK
30578414.json	OK		OK
30644449.json	OK		OK
30679807.json	OK		OK
30842595.json	OK		OK
31686003.json	OK		OK
31806885.json	OK		OK
31836872.json	ERROR		OK
31937935.json	OK		OK
32005979.json	OK		OK
32020081.json	OK		OK
32042144.json	ERROR		OK
32699292.json	OK		OK


## Obtaining the Clustering with PMIDs

In [78]:
import gzip
import json
import logging
import Levenshtein
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig
from pysrc.papers.db.loaders import Loaders

In [35]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [79]:
GROUPED_REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/grouped_refs_validated/'
REFS_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/refs_validated/'
PUBTRENDS_EXPORT_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/pubtrends_export/'
CLUSTERING_FOLDER = '/home/willenjoy/work/pubtrends/nature_reviews/clustering/'

LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD = 5

In [37]:
PUBTRENDS_CONFIG = PubtrendsConfig(test=False)


def reload_exported_analyzer(path_to_archive, source='Pubmed'):
    """
    Load analysis data from json.gz archive generated by PubTrends.
    """
    with gzip.open(path_to_archive, 'rt', encoding='UTF-8') as zipfile:
        data = json.load(zipfile)

    loader, url_prefix = Loaders.get_loader_and_url_prefix(source, PUBTRENDS_CONFIG)
    analyzer = KeyPaperAnalyzer(loader, PUBTRENDS_CONFIG)
    analyzer.init(data)
    
    return analyzer

In [38]:
def reference_index(analyzer, review_pmid):
    reference_pmids = list(analyzer.citations_graph.successors(review_pmid))
    pubmed_data = analyzer.df[analyzer.df['id'].isin(reference_pmids)]
    pubmed_ref_search_index = {v.lower(): k for k, v in zip(pubmed_data['id'], pubmed_data['title'])}
    return pubmed_ref_search_index

In [85]:
def grobid2pmid(refs, reference_index):
    mapping = {}
    ref_mapped = {k: False for k in reference_index.keys()}

    for key, ref in refs.items():
        if ref.lower() in reference_index:
            pmid = reference_index[ref.lower()]
            mapping[key] = int(pmid)
            
    refs_left = {key: ref for key, ref in refs.items() if key not in mapping}
    for pubmed_ref, mapped in ref_mapped.items():
        if not mapped:
            for grobid_id, ref in refs_left.items():
                distance = Levenshtein.distance(ref.lower(), pubmed_ref.lower())
                if distance < LEVENSHTEIN_TITLE_DISTANCE_THRESHOLD:
                    print(ref)
                    print(pubmed_ref)
                    match = input('Are these the same?')
                    if match == 'Y':
                        refs[grobid_id] = pubmed_ref
            
    return mapping, refs

In [40]:
def export_to_json(data, filename):
    with open(filename, 'w') as f:
        json.dump(data, f, indent=4)

In [41]:
def pmid_clustering(grouped_refs, pmid_mapping, prefix=''):
    clustering = []
    for el in grouped_refs:
        if isinstance(el, dict):
            new_element = {
                'title': el['title'],
                'references': pmid_clustering(el['references'], pmid_mapping, prefix='> ')
            }
        else:
            new_element = None
            if el[0]:  # Some references do not have IDs due to parsing error
                grobid_id = el[0][1:]  # '#b0' -> 'b0'
                if grobid_id in pmid_mapping:
                    new_element = pmid_mapping[grobid_id]
    
        if new_element:
            clustering.append(new_element)
            
    return clustering

In [87]:
def obtain_clustering(file, save_clustering=True, save_references=False):
    pmid, ext = os.path.splitext(file)
    
    logging.info(f'{pmid}: loading file with grouped references')
    filename = os.path.join(GROUPED_REFS_FOLDER, file)
    with open(filename, 'r') as f:
        data = json.load(f)

    logging.info(f'{pmid}: loading file with PubTrends analyzer')
    analysis_file = os.path.join(PUBTRENDS_EXPORT_FOLDER, f'{pmid}.json.gz')
    analyzer = reload_exported_analyzer(analysis_file)
    
    logging.info(f'{pmid}: building reference index for matching titles and PMIDs')
    ref_index = reference_index(analyzer, pmid)
    
    logging.info(f'{pmid}: loading file with references')
    refs_file = os.path.join(REFS_FOLDER, file)
    with open(refs_file, 'r') as f:
        refs = json.load(f)
        
    logging.info(f'{pmid}: building mapping between Grobid reference IDs and PMIDs')
    mapping, fixed_refs = grobid2pmid(refs, ref_index)
    n_pubmed_refs = analyzer.citations_graph.out_degree(pmid)
    full_mapping = len(mapping) == n_pubmed_refs
    print(f'{pmid}: {"[100%] " if full_mapping else ""}{len(mapping)} / {n_pubmed_refs} references mapped')
    
    logging.info(f'{pmid}: building clustering with PMIDs instead of Grobid IDs')
    clustering = pmid_clustering(data, mapping)
    clustering_file = os.path.join(CLUSTERING_FOLDER, file)
    
    if save_clustering:
        logging.info(f'{pmid}: exporting clustering')
        export_to_json(clustering, clustering_file)
        
    if save_references:
        logging.info(f'{pmid}: exporting references')
        export_to_json(fixed_refs, refs_file)

    logging.info(f'{pmid}: done\n')
    return len(mapping), n_pubmed_refs

In [93]:
if not os.path.exists(CLUSTERING_FOLDER):
    os.mkdir(CLUSTERING_FOLDER)

refs_mapped = 0
refs_total = 0
for file in sorted(os.listdir(GROUPED_REFS_FOLDER)):
    new_refs_mapped, new_refs_total = obtain_clustering(file)
    refs_mapped += new_refs_mapped
    refs_total += new_refs_total

2021-02-06 23:39:19,516 INFO: 26580716: loading file with grouped references
2021-02-06 23:39:19,517 INFO: 26580716: loading file with PubTrends analyzer
2021-02-06 23:39:20,224 INFO: 26580716: building reference index for matching titles and PMIDs
2021-02-06 23:39:20,227 INFO: 26580716: loading file with references
2021-02-06 23:39:20,233 INFO: 26580716: building mapping between Grobid reference IDs and PMIDs
2021-02-06 23:39:20,369 INFO: 26580716: building clustering with PMIDs instead of Grobid IDs
2021-02-06 23:39:20,371 INFO: 26580716: exporting clustering
2021-02-06 23:39:20,374 INFO: 26580716: done

2021-02-06 23:39:20,410 INFO: 26580717: loading file with grouped references
2021-02-06 23:39:20,413 INFO: 26580717: loading file with PubTrends analyzer


26580716: 126 / 152 references mapped


2021-02-06 23:39:21,351 INFO: 26580717: building reference index for matching titles and PMIDs
2021-02-06 23:39:21,355 INFO: 26580717: loading file with references
2021-02-06 23:39:21,357 INFO: 26580717: building mapping between Grobid reference IDs and PMIDs
2021-02-06 23:39:21,394 INFO: 26580717: building clustering with PMIDs instead of Grobid IDs
2021-02-06 23:39:21,397 INFO: 26580717: exporting clustering
2021-02-06 23:39:21,402 INFO: 26580717: done

2021-02-06 23:39:21,455 INFO: 26656254: loading file with grouped references
2021-02-06 23:39:21,459 INFO: 26656254: loading file with PubTrends analyzer


26580717: 86 / 91 references mapped


2021-02-06 23:39:22,252 INFO: 26656254: building reference index for matching titles and PMIDs
2021-02-06 23:39:22,256 INFO: 26656254: loading file with references
2021-02-06 23:39:22,259 INFO: 26656254: building mapping between Grobid reference IDs and PMIDs
2021-02-06 23:39:22,463 INFO: 26656254: building clustering with PMIDs instead of Grobid IDs
2021-02-06 23:39:22,465 INFO: 26656254: exporting clustering
2021-02-06 23:39:22,467 INFO: 26656254: done

2021-02-06 23:39:22,500 INFO: 26667849: loading file with grouped references
2021-02-06 23:39:22,502 INFO: 26667849: loading file with PubTrends analyzer


...

32020081: 170 / 178 references mapped


2021-02-06 23:39:59,662 INFO: 32042144: building reference index for matching titles and PMIDs
2021-02-06 23:39:59,669 INFO: 32042144: loading file with references
2021-02-06 23:39:59,673 INFO: 32042144: building mapping between Grobid reference IDs and PMIDs
2021-02-06 23:39:59,762 INFO: 32042144: building clustering with PMIDs instead of Grobid IDs
2021-02-06 23:39:59,765 INFO: 32042144: exporting clustering
2021-02-06 23:39:59,768 INFO: 32042144: done

2021-02-06 23:39:59,807 INFO: 32699292: loading file with grouped references
2021-02-06 23:39:59,809 INFO: 32699292: loading file with PubTrends analyzer


32042144: 143 / 204 references mapped


2021-02-06 23:40:00,509 INFO: 32699292: building reference index for matching titles and PMIDs
2021-02-06 23:40:00,513 INFO: 32699292: loading file with references
2021-02-06 23:40:00,517 INFO: 32699292: building mapping between Grobid reference IDs and PMIDs
2021-02-06 23:40:00,584 INFO: 32699292: building clustering with PMIDs instead of Grobid IDs
2021-02-06 23:40:00,588 INFO: 32699292: exporting clustering
2021-02-06 23:40:00,592 INFO: 32699292: done



32699292: 144 / 153 references mapped


In [91]:
refs_mapped / refs_total

0.9035258814703676

## Bulk PubTrends Export

In [ ]:
import gzip
import json
import logging
import os

from pysrc.papers.analyzer import KeyPaperAnalyzer
from pysrc.papers.pubtrends_config import PubtrendsConfig, DEFAULT_ANALYZER_SETTINGS
from pysrc.papers.db.loaders import Loaders
from pysrc.papers.db.search_error import SearchError
from pysrc.papers.plot.plotter import visualize_analysis
from pysrc.papers.progress import Progress
from pysrc.papers.utils import SORT_MOST_CITED, ZOOM_OUT, PAPER_ANALYSIS

In [ ]:
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')

In [ ]:
TARGET_FOLDER = '/home/willenjoy/work/pubtrends/local/pubtrends_export/'

TARGET_PMIDS = [26667849, 26678314, 26688349, 26688350, 26580716, 26580717, 26656254, 26675821, 27834397, 27834398,
                27890914, 27916977, 27677859, 27677860, 27904142, 28003656, 29147025, 29170536, 28853444, 28920587,
                28792006, 28852220, 29213134, 29321682, 30578414, 30842595, 30644449, 30679807, 30108335, 30390028,
                30459365, 30467385, 31686003, 31806885, 31836872, 32005979, 31937935, 32020081, 32042144, 32699292]

SOURCE = 'Pubmed'
LIMIT = 500

In [ ]:
def export_analysis(pmid):
    logging.info(f'Started analysis for PMID {pmid}')
    
    ids = [pmid]
    query = f'Paper ID: {pmid}'
    
    # extracted from 'analyze_id_list' Celery task
    config = PubtrendsConfig(test=False)
    loader = Loaders.get_loader(SOURCE, config)
    analyzer = KeyPaperAnalyzer(loader, config)
    settings = DEFAULT_ANALYZER_SETTINGS
    try:
        # Fetch references at first
        ids = ids + analyzer.load_references(
            ids[0], limit=LIMIT
        )
        # And then expand
        ids = analyzer.expand_ids(
            ids, limit=LIMIT,
            expand_steps=settings.EXPAND_STEPS, expand_citations_sigma=settings.EXPAND_CITATIONS_SIGMA,
            expand_similarity_threshold=settings.EXPAND_SIMILARITY_THRESHOLD,
            current=1, task=None
        )

        analyzer.analyze_papers(ids, query, task=None)
    finally:
        loader.close_connection()

    dump = analyzer.dump()
    
    # export as JSON
    path = os.path.join(TARGET_FOLDER, f'{pmid}.json.gz')
    with gzip.open(path, 'w') as f:
        f.write(json.dumps(dump).encode('utf-8'))
    
    logging.info(f'Finished analysis for PMID {pmid}\n')

In [ ]:
TARGET_PMIDS = [49, 58, 59, 64]

In [ ]:
for pmid in TARGET_PMIDS:
    export_analysis(pmid)